# M02 – Magic Methods & Special Methods

**PCPP Alignment**: Section 1 (1.2) – Perform Python core syntax operations  
**Professional Focus**: Making objects behave like built-in types, operator overloading.

---

## Learning Outcomes

- Implement comparison magic methods (`__eq__`, `__lt__`, etc.)
- Implement numeric magic methods (`__add__`, `__sub__`, `__mul__`, etc.)
- Implement type conversion methods (`__str__`, `__repr__`, `__int__`, etc.)
- Implement attribute access methods (`__getattr__`, `__setattr__`)
- Implement container methods (`__getitem__`, `__len__`, `__iter__`)
- Use `@functools.total_ordering` to reduce boilerplate

---

## Table of Contents

1. Comparison Methods (PCPP 1.2)
2. Numeric Methods (PCPP 1.2)
3. Type Conversion Methods (PCPP 1.2)
4. Attribute Access Methods (PCPP 1.2)
5. Container Methods (PCPP 1.2)
6. Common Patterns and Best Practices
7. Practice Exercises

## 1. Comparison Methods (PCPP 1.2)

Comparison methods allow objects to support comparison operators: `==`, `!=`, `<`, `<=`, `>`, `>=`

In [ ]:
from functools import total_ordering

@total_ordering
class Point:
    """Point class with comparison methods."""
    
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y
    
    def __eq__(self, other: object) -> bool:
        """Equality: point1 == point2"""
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y
    
    def __lt__(self, other: 'Point') -> bool:
        """Less-than: point1 < point2 (by distance from origin)"""
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x ** 2 + self.y ** 2) < (other.x ** 2 + other.y ** 2)
    
    def __str__(self) -> str:
        return f"Point({self.x}, {self.y})"

# Test comparison methods
p1 = Point(1, 2)
p2 = Point(3, 4)
p3 = Point(1, 2)

print(f"p1 == p3: {p1 == p3}")
print(f"p1 < p2: {p1 < p2}")
print(f"p1 <= p2: {p1 <= p2}")  # Works because of @total_ordering
print(f"p1 > p2: {p1 > p2}")

**Additional Example**: Sort Points with @total_ordering.


In [ ]:
from functools import total_ordering

@total_ordering
class Score:
    def __init__(self, value):
        self.value = value
    def __eq__(self, other):
        return self.value == other.value
    def __lt__(self, other):
        return self.value < other.value

print(sorted([Score(30), Score(10), Score(20)], key=lambda s: s.value))


## 2. Numeric Methods (PCPP 1.2)

Numeric methods allow objects to support arithmetic operations: `+`, `-`, `*`, `/`, `abs()`, etc.

In [ ]:
class Vector:
    """Vector class with numeric methods."""
    
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y
    
    def __add__(self, other: 'Vector') -> 'Vector':
        """Addition: vector1 + vector2"""
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)
    
    def __mul__(self, scalar: float) -> 'Vector':
        """Scalar multiplication: vector * scalar"""
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector(self.x * scalar, self.y * scalar)
    
    def __rmul__(self, scalar: float) -> 'Vector':
        """Right-hand multiplication: scalar * vector"""
        return self.__mul__(scalar)
    
    def __abs__(self) -> float:
        """Absolute value (magnitude)"""
        return (self.x ** 2 + self.y ** 2) ** 0.5
    
    def __str__(self) -> str:
        return f"Vector({self.x}, {self.y})"

# Test numeric methods
v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(f"v1 + v2 = {v1 + v2}")
print(f"v1 * 2 = {v1 * 2}")
print(f"3 * v1 = {3 * v1}")  # Right-hand multiplication
print(f"|v1| = {abs(v1)}")

**Additional Example**: __sub__ for vector subtraction.


In [ ]:
class V:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __sub__(self, other):
        return V(self.x - other.x, self.y - other.y)
    def __str__(self):
        return f'V({self.x},{self.y})'

print(V(5, 7) - V(1, 2))


## 3. Type Conversion Methods (PCPP 1.2)

Type conversion methods allow objects to be converted to other types: `str()`, `int()`, `float()`, `bool()`

In [ ]:
class Fraction:
    """Fraction class with type conversion methods."""
    
    def __init__(self, numerator: int, denominator: int = 1) -> None:
        if denominator == 0:
            raise ValueError("Denominator cannot be zero")
        self.numerator = numerator
        self.denominator = denominator
    
    def __float__(self) -> float:
        """Convert to float"""
        return self.numerator / self.denominator
    
    def __int__(self) -> int:
        """Convert to int (truncates)"""
        return self.numerator // self.denominator
    
    def __str__(self) -> str:
        """User-friendly string"""
        return f"{self.numerator}/{self.denominator}"
    
    def __repr__(self) -> str:
        """Developer representation (should be evaluable)"""
        return f"Fraction({self.numerator}, {self.denominator})"
    
    def __bool__(self) -> bool:
        """Truthiness (non-zero is truthy)"""
        return self.numerator != 0

# Test type conversion
f = Fraction(3, 4)
print(f"Fraction: {f}")
print(f"float(f): {float(f)}")
print(f"int(f): {int(f)}")
print(f"bool(f): {bool(f)}")
print(f"bool(Fraction(0, 1)): {bool(Fraction(0, 1))}")

**Additional Example**: __repr__ for debugging.


In [ ]:
class SKU:
    def __init__(self, code):
        self.code = code
    def __repr__(self):
        return f"SKU({self.code!r})"

print([SKU('A1'), SKU('B2')])


## 4. Attribute Access Methods (PCPP 1.2)

Attribute access methods control how attributes are accessed and set: `__getattr__`, `__setattr__`, `__getattribute__`

In [ ]:
class Config:
    """Config class with dynamic attribute access."""
    
    def __init__(self) -> None:
        self._data: dict = {}
        self._defaults = {'host': 'localhost', 'port': 8080}
    
    def __getattr__(self, name: str):
        """Called when attribute not found via normal lookup"""
        if name in self._defaults:
            return self._defaults[name]
        raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")
    
    def __setattr__(self, name: str, value) -> None:
        """Called when setting any attribute"""
        if name.startswith('_'):
            super().__setattr__(name, value)
        else:
            if not hasattr(self, '_data'):
                super().__setattr__('_data', {})
            self._data[name] = value

# Test attribute access
config = Config()
print(f"config.host: {config.host}")  # Uses __getattr__
config.port = 9000  # Uses __setattr__
print(f"config.port: {config.port}")

**Additional Example**: __setattr__ validation.


In [ ]:
class Positive:
    def __setattr__(self, name, value):
        if name == 'x' and value < 0:
            raise ValueError('x must be >= 0')
        super().__setattr__(name, value)

p = Positive()
p.x = 10
print(p.x)


## 5. Container Methods (PCPP 1.2)

Container methods make objects behave like lists or dicts: `__getitem__`, `__len__`, `__contains__`, `__iter__`

In [ ]:
class Deck:
    """Deck class demonstrating container methods."""
    
    def __init__(self, cards: list[str] | None = None) -> None:
        if cards is None:
            suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
            ranks = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K']
            cards = [f"{rank} of {suit}" for suit in suits for rank in ranks]
        self._cards = cards
    
    def __getitem__(self, index: int | slice) -> str | list[str]:
        """Make deck subscriptable"""
        return self._cards[index]
    
    def __len__(self) -> int:
        """Return number of cards"""
        return len(self._cards)
    
    def __contains__(self, card: str) -> bool:
        """Check if card is in deck"""
        return card in self._cards
    
    def __iter__(self):
        """Make deck iterable"""
        return iter(self._cards)

# Test container methods
deck = Deck()
print(f"Deck length: {len(deck)}")
print(f"First card: {deck[0]}")
print(f"'Ace of Spades' in deck: {'Ace of Spades' in deck}")
print(f"First 5 cards: {deck[:5]}")
print("\nIterating:")
for i, card in enumerate(deck[:5]):
    print(f"  {i+1}. {card}")

**Additional Example**: __contains__ for membership.


In [ ]:
class Bag:
    def __init__(self, items):
        self._items = list(items)
    def __contains__(self, item):
        return item in self._items

b = Bag(['pen', 'pad'])
print('pen' in b, 'phone' in b)


## 6. Common Patterns and Best Practices

### Returning NotImplemented

When a magic method doesn't support a particular operation, return `NotImplemented` (not `NotImplementedError`). This allows Python to try the right-hand method.

In [ ]:
class Number:
    def __init__(self, value: int) -> None:
        self.value = value
    
    def __add__(self, other):
        if isinstance(other, Number):
            return Number(self.value + other.value)
        # Return NotImplemented, not raise exception
        return NotImplemented

n1 = Number(5)
n2 = Number(3)
print(f"n1 + n2 = {(n1 + n2).value}")

# If we try n1 + 10, Python will try 10.__radd__(n1)
# If that also returns NotImplemented, TypeError is raised

### __str__ vs __repr__

- `__str__`: User-friendly, readable representation
- `__repr__`: Developer-friendly, ideally evaluable (can recreate object)

In [ ]:
class Person:
    def __init__(self, name: str, age: int) -> None:
        self.name = name
        self.age = age
    
    def __str__(self) -> str:
        return f"{self.name}, {self.age} years old"
    
    def __repr__(self) -> str:
        return f"Person(name='{self.name}', age={self.age})"

p = Person("Alice", 30)
print(f"str(p): {str(p)}")      # User-friendly
print(f"repr(p): {repr(p)}")    # Developer-friendly

**Additional Example**: __str__ vs __repr__ side by side.


In [ ]:
class Product:
    def __init__(self, name, price):
        self.name, self.price = name, price
    def __str__(self):
        return f'{self.name} (${self.price})'
    def __repr__(self):
        return f"Product({self.name!r}, {self.price})"

p = Product('Widget', 9.99)
print(str(p), '|', repr(p))


## 7. Practice Exercises

Complete the exercises in `practice/practice_02_magic_methods.py`